# Momentum cross-sectional en divisas

**Notebooks previos:** Backtest-Corregido, Beta-Anclado-OU, Remuestreo-Costos, Amplitud, Eventos, Datos-Diarios

## Estructura de este Notebook
1. **Fase 0: Datos y bloques.** Reutilizamos `data_diaria.pkl` con el corte en 2019 (diseño) y 2020+ (prueba, sin tocar).
2. **Fase 1: Señal y portafolio.** Retorno de cada moneda frente al dólar en los últimos 12 meses (saltando el último mes). Cada mes: largo en las 3 monedas con mayor retorno, corto en las 3 con menor retorno, pesos iguales (neutral al dólar). Regla fijada de antemano: 12-1 meses, 3 + 3, rebalanceo mensual.
3. **Fase 2: Resultados en el bloque de diseño.** Retorno anual, Sharpe, caída máxima, rotación y sensibilidad a costos (1, 3 y 5 pb por pata). Retorno por año, para ver si depende de un solo periodo.
4. **Fase 3: Placebo.** El mismo procedimiento con señales aleatorias (monedas ordenadas al azar), para ver cuánto Sharpe produce el azar con esta muestra.
5. **Fase 4: Evaluación única en 2020+.** La regla de la Fase 1, sin cambiarla, una sola vez.
6. **Conclusión.**

## Fase 0: Datos y bloques

Reutilizamos los datos diarios de *Datos-Diarios* (`data_diaria.pkl`, 18 divisas desde 2010) y el mismo corte: **diseño hasta 2019** y **prueba desde 2020**. El archivo contiene ambos bloques, pero aquí solo se crea la variable del bloque de diseño; el de prueba no se carga hasta la Fase 4, para no tener la tentación de mirarlo.

*Entrega:* `data_sel`.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings
from scipy import stats
warnings.filterwarnings('ignore')

FECHA_CORTE = pd.Timestamp('2019-12-31')
data = pd.read_pickle('data_diaria.pkl')
data_sel = data[data.index <= FECHA_CORTE]       # bloque de diseño. El de prueba NO se crea hasta la Fase 4
del data
print(data_sel.shape, data_sel.index.min().date(), '->', data_sel.index.max().date())

(2607, 18) 2010-01-01 -> 2019-12-31


## Fase 1: Señal y portafolio

**Idea.** Es lo opuesto a la reversión: las monedas que han subido frente al dólar durante el último año tienden a seguir subiendo, y las que han bajado, a seguir bajando. Cada mes ordenamos las 18 monedas por su rendimiento de los últimos 12 meses, **saltando el último mes** (que suele mostrar una reversión de corto plazo), y armamos un portafolio neutral al dólar: **largo en las 3 más fuertes y corto en las 3 más débiles**, con pesos iguales.

**Regla fijada de antemano** (la más estándar de la literatura, no la que mejor se vea): lookback de 12 meses, salto de 1, 3 + 3 monedas, rebalanceo mensual. No se ajustará nada después.

**Convención.** Todas las series se expresan como *dólares por unidad de moneda* (si la moneda sube, el valor sube). Eso exige invertir las cotizadas como USDxxx; Euro, Libra y los dólares australiano y neozelandés ya vienen en esa forma.

**Sin ver el futuro.** La señal al cierre del mes *m* usa solo precios hasta el cierre del mes *m − 1*; la posición se mantiene durante el mes *m + 1*.

*Entrega:* función `momentum_mensual`, que devuelve el retorno bruto mensual y la rotación, y verificaciones de que el portafolio está bien armado. Aquí **no** miramos rendimientos: eso es la Fase 2.

In [3]:
PREC_EN_USD = ['Euro', 'Libra', 'Dolar_AUS', 'Dolar_NZ']          # ya cotizadas como dólares por unidad

def a_dolares_por_moneda(px):
    fx = px.copy()
    for c in fx.columns:
        if c not in PREC_EN_USD:
            fx[c] = 1.0 / fx[c]
    return fx

fx = a_dolares_por_moneda(data_sel)

In [8]:
def momentum_mensual(fx, lookback=12, skip=1, n_side=3):
    """Largo en las n_side monedas con mayor retorno [t-lookback, t-skip], corto en las n_side con menor retorno."""
    pm = fx.groupby(fx.index.to_period('M')).last()                 # precio de cierre de cada mes
    lp = np.log(pm); r = lp.diff()                                   # r[m]: retorno del mes m
    S = lp.shift(skip) - lp.shift(lookback)                          # señal conocida al cierre del mes m
    rank = S.rank(axis=1, method='first', ascending=False)
    W = pd.DataFrame(0.0, index=S.index, columns=S.columns)
    W[rank <= n_side] = 1.0 / n_side
    W[rank > S.notna().sum(axis=1).values[:, None] - n_side] = -1.0 / n_side
    W[S.isna().any(axis=1)] = 0.0                                    # sin historia suficiente: sin posición
    ret = (W.shift(1) * r).sum(axis=1)                               # los pesos del cierre m-1 ganan el retorno del mes m
    rot = (W - W.shift(1)).abs().sum(axis=1)                         # cuánto se opera cada mes
    out = pd.DataFrame({'ret_bruto': ret, 'rotacion': rot})
    return out.iloc[lookback + 1:], W

mom, W = momentum_mensual(fx)

In [9]:
def momentum_mensual(fx, lookback=12, skip=1, n_side=3):
    """Largo en las n_side monedas con mayor retorno [t-lookback, t-skip], corto en las n_side con menor retorno."""
    pm = fx.groupby(fx.index.to_period('M')).last()                 # precio de cierre de cada mes
    lp = np.log(pm); r = lp.diff()                                   # r[m]: retorno del mes m
    S = lp.shift(skip) - lp.shift(lookback)                          # señal conocida al cierre del mes m
    rank = S.rank(axis=1, method='first', ascending=False)
    W = pd.DataFrame(0.0, index=S.index, columns=S.columns)
    W[rank <= n_side] = 1.0 / n_side
    W[rank > S.notna().sum(axis=1).values[:, None] - n_side] = -1.0 / n_side
    W[S.isna().any(axis=1)] = 0.0                                    # sin historia suficiente: sin posición
    ret = (W.shift(1) * r).sum(axis=1)                               # los pesos del cierre m-1 ganan el retorno del mes m
    rot = (W - W.shift(1)).abs().sum(axis=1)                         # cuánto se opera cada mes
    out = pd.DataFrame({'ret_bruto': ret, 'rotacion': rot})
    return out.iloc[lookback + 1:], W

mom, W = momentum_mensual(fx)

In [11]:
Wv = W.loc[mom.index]
print("Meses del portafolio:", len(mom), "|", mom.index.min(), "->", mom.index.max())
print("Suma de pesos (debe ser 0):", Wv.sum(axis=1).abs().max())
print("Exposición bruta (debe ser 2):", Wv.abs().sum(axis=1).round(6).unique())
print("Monedas largas / cortas por mes (3 / 3):", (Wv > 0).sum(axis=1).unique(), (Wv < 0).sum(axis=1).unique())
print("Rotación media mensual (sobre exposición 2):", mom.rotacion.mean().round(3))

Meses del portafolio: 107 | 2011-02 -> 2019-12
Suma de pesos (debe ser 0): 1.1102230246251565e-16
Exposición bruta (debe ser 2): [2.]
Monedas largas / cortas por mes (3 / 3): [3] [3]
Rotación media mensual (sobre exposición 2): 1.265


In [12]:
print("Fracción de meses largo / corto, por moneda:")
print(pd.DataFrame({'largo': (Wv > 0).mean(), 'corto': (Wv < 0).mean()}).round(2).sort_values('corto', ascending=False))

Fracción de meses largo / corto, por moneda:
            largo  corto
Ticker                  
Rand_ZA      0.12   0.51
Corona_SE    0.14   0.31
Yen          0.33   0.26
Peso_MX      0.10   0.26
Dolar_AUS    0.15   0.25
Forinto_HU   0.04   0.21
Corona_NO    0.03   0.20
Libra        0.21   0.19
Dolar_NZ     0.23   0.16
Zloty_PL     0.15   0.16
Baht_TH      0.30   0.10
Corona_CZ    0.17   0.09
Won_KR       0.22   0.08
Dolar_CAN    0.06   0.07
Shekel_IL    0.28   0.06
Euro         0.12   0.05
Franco_SUI   0.25   0.03
Dolar_SG     0.09   0.01


## Fase 2: Resultados en el bloque de diseño

Aquí medimos cómo habría funcionado la regla de la Fase 1 en 2011–2019, **sin cambiarla**. Se reportan:
- **Retorno anual, volatilidad, Sharpe y estadístico t** del portafolio (el t es la medida de si el resultado se distingue del azar; con ~107 meses, un t cercano a 2 es el mínimo para empezar a tomarlo en serio).
- **Caída máxima** (la peor pérdida acumulada desde un máximo).
- **Sensibilidad a costos:** 0, 1, 3, 5 y 10 pb por pata y por lado, aplicados a la rotación mensual. 1 pb es el supuesto de los notebooks anteriores; 3–10 pb cubren los spreads probables de las monedas emergentes que dominan el portafolio.
- **Retorno por año**, para ver si el resultado depende de un solo periodo.
- **Contribución por moneda**, para ver si el resultado depende de una sola moneda (por ejemplo, el Rand).

Los retornos son de un portafolio autofinanciado (largo 1, corto 1), así que no se resta tasa libre de riesgo.

In [13]:
def metricas(r):
    mu, sd = r.mean() * 12, r.std() * np.sqrt(12)
    cum = r.cumsum()
    return pd.Series({'ret_anual': mu, 'vol_anual': sd, 'sharpe': mu / sd,
                      't': r.mean() / (r.std() / np.sqrt(len(r))),
                      'caida_max': (cum - cum.cummax()).min(), 'meses>0': (r > 0).mean()})

filas = {}
for bps in (0, 1, 3, 5, 10):
    filas[f'{bps} pb por pata'] = metricas(mom.ret_bruto - mom.rotacion * bps / 1e4)
display(pd.DataFrame(filas).T.round(3))

,ret_anual,vol_anual,sharpe,t,caida_max,meses>0
0 pb por pata,-0.002,0.075,-0.027,-0.082,-0.239,0.477
1 pb por pata,-0.004,0.075,-0.048,-0.142,-0.243,0.467
3 pb por pata,-0.007,0.075,-0.088,-0.263,-0.251,0.449
5 pb por pata,-0.010,0.075,-0.129,-0.384,-0.258,0.439
10 pb por pata,-0.017,0.075,-0.230,-0.685,-0.278,0.439


In [14]:
neto3 = mom.ret_bruto - mom.rotacion * 3 / 1e4
por_anio = neto3.groupby(neto3.index.year).agg(['sum', lambda x: (x > 0).mean()])
por_anio.columns = ['retorno', 'meses>0']
display(por_anio.round(3).T)

Date,2011,2012,2013,2014,2015,2016,2017,2018,2019
retorno,-0.055,-0.08,0.114,-0.012,0.083,0.041,-0.156,-0.047,0.053
meses>0,0.455,0.25,0.750,0.333,0.500,0.500,0.250,0.417,0.583


In [16]:
pm = fx.groupby(fx.index.to_period('M')).last(); r = np.log(pm).diff()
contrib = (W.shift(1) * r).loc[mom.index].sum().sort_values()          # contribución acumulada, bruta
print("Contribución acumulada por moneda (suma de retornos mensuales):")
display(contrib.round(3).to_frame('contribucion'))
print("Peso de cada moneda en la contribución absoluta (las 3 mayores):")
print((contrib.abs() / contrib.abs().sum()).sort_values(ascending=False).head(3).round(2))

Contribución acumulada por moneda (suma de retornos mensuales):


,contribucion
Ticker,
Dolar_NZ,-0.082
Won_KR,-0.074
Dolar_SG,-0.041
Dolar_CAN,-0.038
Peso_MX,-0.038
Euro,-0.025
Franco_SUI,-0.023
Forinto_HU,-0.016
Libra,-0.013


Peso de cada moneda en la contribución absoluta (las 3 mayores):
Ticker
Rand_ZA     0.13
Dolar_NZ    0.12
Yen         0.11
dtype: float64


## Conclusión de Momentum-FX

**Qué se hizo.** Se probó la contraparte de la reversión: un portafolio neutral al dólar que cada mes compra las 3 monedas con mayor rendimiento de los últimos 12 meses (saltando el último) y vende las 3 con menor rendimiento, con 18 divisas, en 2011–2019. La regla (12-1, 3 + 3, rebalanceo mensual) se fijó antes de ver resultados y no se ajustó.

**Qué se encontró.**
1. Sin costos, el retorno anual bruto es −0.2% (Sharpe −0.03, t = −0.08): no hay señal distinguible del azar.
2. Con 1, 3, 5 y 10 pb por pata el retorno neto es −0.4%, −0.7%, −1.0% y −1.7% anual; la rotación (1.27 por mes) es baja, pero no hay ganancia que preservar.
3. El resultado por año es mixto (4 años positivos y 5 negativos, de −16% a +11%).
4. El portafolio estuvo dominado por la tendencia del periodo: corto en Rand (51% de los meses), Corona sueca y Peso, largo en monedas refugio (Yen, Baht, Franco).

**Qué significa.** La estrategia de momentum transversal estándar no produce un retorno distinguible de cero en esta muestra, ni siquiera antes de costos. A diferencia del pairs trading, aquí el obstáculo no son los costos sino la ausencia de señal.

**Decisión de diseño.** La regla no pasó en el bloque de diseño, así que no se evaluó el bloque 2020–2026, que queda sin contaminar para futuras propuestas.

**Limitaciones.** Una sola regla (la más estándar); 107 meses (un Sharpe de 0.3 habría dado t ≈ 1, así que la muestra no permite detectar rendimientos modestos); pesos iguales sin ajustar por volatilidad; sin tasas de interés (no se evaluó carry).